# Ollama · do modelo local ao fluxo com estado
## Banco BV · AI Experts II · Laboratório guiado
**Prof. Ioannis Eleftheriou · 25/09/2026**

> **Tese da aula** — Rodar o modelo perto dos dados muda a infraestrutura. Não elimina a necessidade de contratos, avaliação, permissões e explicações.
## Rodando um agente documental sem API paga

> e se eu quiser rodar tudo localmente, usando modelos gratuitos, sem depender de uma chave da OpenAI?

A resposta prática será:

- **Ollama** para rodar modelos locais;
- **LangChain** para carregar PDF, gerar embeddings e consultar o LLM;
- **ChromaDB** para persistir o índice vetorial;
- **LangGraph** para orquestrar o workflow de decisão e resposta.

A proposta não é fingir que modelo local pequeno é igual a modelo frontier. Não é.

A proposta é mostrar uma arquitetura que funciona, que ensina bem e que pode ser executada em máquina comum para estudo, prototipação e demonstração.

## Mapa da aula

```text
1. Entender o papel do Ollama em um workflow de RAG local
2. Configurar modelos gratuitos para chat e embeddings
3. Carregar o regulamento financeiro em PDF
4. Dividir o documento em chunks
5. Criar um vector store local com ChromaDB + OllamaEmbeddings
6. Construir uma capacidade RAG local
7. Orquestrar o fluxo com LangGraph
8. Testar cenários: pergunta documental, taxas e fora de escopo
9. Encapsular a lógica em um módulo reutilizável
10. Propor exercícios de evolução
```

Arquitetura final:

```text
Usuário
  |
  v
LangGraph Workflow
  |
  +--> classificar_intencao  -> Ollama Chat
  |
  +--> recuperar_contexto    -> ChromaDB + Ollama Embeddings
  |
  +--> gerar_resposta        -> Ollama Chat
  |
  +--> formatar_saida        -> resposta + fontes + páginas
```

Repare na diferença didática:

- no RAG simples, chamamos uma chain;
- no agente genérico, deixamos o modelo decidir muita coisa;
- no LangGraph, desenhamos explicitamente os estados e nós do workflow.

Isso é muito útil em contexto corporativo, porque decisão explícita é mais fácil de auditar.

## Objetivos de aprendizagem

Ao final da aula, você será capaz de:

1. Explicar a diferença entre rodar RAG com API externa e RAG com modelo local.
2. Usar `ChatOllama` para geração de respostas em português.
3. Usar `OllamaEmbeddings` para criar embeddings locais.
4. Persistir um índice vetorial com ChromaDB.
5. Criar um workflow LangGraph com estado tipado.
6. Separar classificação, recuperação, geração e formatação em nós diferentes.
7. Identificar limitações reais de modelos locais pequenos.

A régua aqui é simples: não basta a resposta sair bonita. Ela precisa ter fonte, página, escopo e uma arquitetura que o aluno consiga explicar.

## Parte 0 — Pré-requisitos do Ollama

Antes de executar as células Python, instale o Ollama e baixe os modelos.

Site oficial:

<https://ollama.com/download>

No terminal:

```bash
ollama pull llama3.2:3b
ollama pull nomic-embed-text
```

Modelo de chat sugerido:

| Modelo | Papel | Por que usar |
|---|---|---|
| `llama3.2:3b` | LLM de resposta e classificação | leve, gratuito e suficiente para aula |
| `nomic-embed-text` | embeddings | muito usado com Ollama para busca semântica local |

Se sua máquina tiver mais memória, você pode testar modelos maiores, como `llama3.1:8b`, `mistral` ou `qwen2.5:7b`.

Mas para aula, o melhor modelo é aquele que roda na máquina do aluno. Sofisticação que não executa não serve pra nada.

In [1]:
# ============================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS
# ============================================================
# Execute esta célula se o ambiente ainda não estiver configurado.
# Em ambientes corporativos, prefira instalar pelo terminal:
# pip install -r requirements.txt

%pip install -q langchain langchain-community langchain-chroma langchain-text-splitters langchain-ollama langgraph chromadb pypdf pydantic python-dotenv httpx


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.14.7 requires chromadb~=1.1.0, but you have chromadb 1.5.9 which is incompatible.
crewai 1.14.7 requires openai<3,>=2.30.0, but you have openai 1.58.1 which is incompatible.
crewai 1.14.7 requires opentelemetry-api~=1.34.0, but you have opentelemetry-api 1.43.0 which is incompatible.
crewai 1.14.7 requires opentelemetry-sdk~=1.34.0, but you have opentelemetry-sdk 1.43.0 which is incompatible.
crewai-cli 1.14.7 requires pydantic-settings~=2.10.1, but you have pydantic-settings 2.12.0 which is incompatible.
crewai-core 1.14.7 requires opentelemetry-api~=1.34.0, but you have opentelemetry-api 1.43.0 which is incompatible.
crewai-core 1.14.7 requires opentelemetry-sdk~=1.34.0, but you have opentelemetry-sdk 1.43.0 which is incompatible.
google-adk 2.2.0 requires fastapi<1,>=0.133, but you have fastapi 0.123.9

In [19]:
# ============================================================
# IMPORTS BÁSICOS E DIAGNÓSTICO DO AMBIENTE
# ============================================================

from __future__ import annotations

import os
import pathlib
from typing import Literal, Optional, TypedDict

import httpx
from pydantic import BaseModel, Field

BASE_DIR = pathlib.Path.cwd()
print(f"Diretório atual: {BASE_DIR}")

# Modelos padrão da aula. Troque aqui se quiser experimentar.
# Acesso direto temporario nesta maquina; alunos com Ollama nativo usam 11434.
OLLAMA_BASE_URL = "http://127.0.0.1:11435"
OLLAMA_CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2:3b")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text")

print("Ollama base URL:", OLLAMA_BASE_URL)
print("Modelo de chat:", OLLAMA_CHAT_MODEL)
print("Modelo de embedding:", OLLAMA_EMBED_MODEL)


Diretório atual: c:\NYX-WORLD\Nyx Continuum\Projects\01-banco-bv\aula-extra-ollama-langgraph
Ollama base URL: http://127.0.0.1:11435
Modelo de chat: llama3.2:3b
Modelo de embedding: nomic-embed-text


### 0.1 Verificando se o Ollama está acessível

O Ollama precisa estar rodando localmente. Em muitas instalações, ele já sobe como serviço. Se não estiver, rode em um terminal:

```bash
ollama serve
```

A célula abaixo chama a API local do Ollama e lista os modelos disponíveis.

In [ ]:
# ============================================================
# DIAGNÓSTICO DO OLLAMA
# ============================================================

def listar_modelos_ollama(base_url: str = OLLAMA_BASE_URL) -> list[str]:
    try:
        resp = httpx.get(f"{base_url}/api/tags", timeout=5)
        resp.raise_for_status()
        dados = resp.json()
        return [m.get("name", "") for m in dados.get("models", [])]
    except Exception as exc:
        raise RuntimeError(
            "Não consegui acessar o Ollama. Verifique se o Ollama está instalado e rodando. "
            "Tente executar: ollama serve"
        ) from exc


modelos = listar_modelos_ollama()
print("Modelos encontrados no Ollama:")
for modelo in modelos:
    print("-", modelo)

# Nomes sem tag equivalem a :latest.
def nome_com_tag(nome: str) -> str:
    return nome if ":" in nome.rsplit("/", 1)[-1] else f"{nome}:latest"

modelos_disponiveis = {nome_com_tag(m) for m in modelos}
faltando = [m for m in [OLLAMA_CHAT_MODEL, OLLAMA_EMBED_MODEL]
            if nome_com_tag(m) not in modelos_disponiveis]
if faltando:
    print("\nModelos esperados ainda não encontrados:", faltando)
    print("Baixe com:")
    for modelo in faltando:
        print(f"ollama pull {modelo}")
else:
    print("\nAmbiente Ollama pronto para a aula.")


## Parte 1 — Preparando o documento

Vamos reutilizar o regulamento financeiro das aulas anteriores.

Caminho esperado:

```text
../Aula 03 - MCP II/data/KNCR_Regulamento_05-2025.pdf
```

Se você estiver rodando este notebook fora do repositório, ajuste `PDF_PATH` para o caminho correto.

A escolha é deliberada: a aula muda a infraestrutura de modelo, não o problema de negócio. Isso facilita comparar OpenAI vs Ollama.

AQUI AJUSTA PRA PASTA ATUAL E TRAZ O ARQUIVO DO GIT PRA ELA...

In [17]:
# ============================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================

PDF_PATH = (BASE_DIR / ".." / "aula-extra-ollama-langgraph" / "data" / "KNCR_Regulamento_05-2025.pdf").resolve()
PERSIST_DIR = (BASE_DIR / "vectorstore" / "chroma_kncr_ollama").resolve()
COLLECTION_NAME = "regulamento_kncr_ollama"

print("PDF:", PDF_PATH)
print("PDF existe?", PDF_PATH.exists())
print("Vector store:", PERSIST_DIR)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF não encontrado em {PDF_PATH}. Ajuste PDF_PATH para o regulamento que deseja consultar."
    )


PDF: C:\NYX-WORLD\Nyx Continuum\Projects\01-banco-bv\aula-extra-ollama-langgraph\data\KNCR_Regulamento_05-2025.pdf
PDF existe? True
Vector store: C:\NYX-WORLD\Nyx Continuum\Projects\01-banco-bv\aula-extra-ollama-langgraph\vectorstore\chroma_kncr_ollama


## Parte 2 — Carregamento e chunking

O RAG começa antes do LLM.

Pipeline documental:

```text
PDF
  -> páginas
  -> chunks
  -> embeddings
  -> vector store
```

Em aula, esse trecho é importante porque mostra que erro de RAG muitas vezes não é erro do modelo. Pode ser:

- PDF mal extraído;
- chunks grandes demais;
- chunks pequenos demais;
- embedding fraco;
- recuperação sem contexto suficiente;
- pergunta fora do escopo.

In [18]:
# ============================================================
# CARREGAMENTO DO PDF E CHUNKING
# ============================================================

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
paginas = loader.load()

print(f"Páginas carregadas: {len(paginas)}")
print("Metadata da primeira página:", paginas[0].metadata)
print("Trecho inicial:\n")
print(paginas[0].page_content[:1000])

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=140,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(paginas)
print(f"Chunks gerados: {len(chunks)}")
print("Exemplo de chunk:\n")
print(chunks[0].page_content[:1000])


Páginas carregadas: 40
Metadata da primeira página: {'producer': 'Microsoft® Word para Microsoft 365', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-05-14T09:27:21-03:00', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_enabled': 'true', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_setdate': '2023-09-21T16:31:37Z', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_method': 'Privileged', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_name': 'Compartilhamento Externo', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_siteid': '591669a0-183f-49a5-98f4-9aa0d0b63d81', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_actionid': 'c9b9016b-0eeb-42ab-b99a-c8596d2c030e', 'msip_label_59f6b450-b779-4ed9-b37e-4a5b0cc9de23_contentbits': '0', 'author': 'i2a Advogados', 'moddate': '2025-05-14T09:27:21-03:00', 'source': 'C:\\NYX-WORLD\\Nyx Continuum\\Projects\\01-banco-bv\\aula-extra-ollama-langgraph\\data\\KNCR_Regulamento_05-2025.pdf', 'total_pages': 40, 'page': 0, 'pag

## Parte 3 — Embeddings locais com Ollama

Nas aulas anteriores, os embeddings vinham de um provedor externo.

Aqui usamos `OllamaEmbeddings` com `nomic-embed-text`.

Isso muda a conversa de custo e privacidade:

| Critério | API externa | Ollama local |
|---|---|---|
| Custo por chamada | existe | zero por chamada |
| Latência | depende da rede | depende da máquina |
| Privacidade | envia texto ao provedor | fica local |
| Qualidade | geralmente maior | depende do modelo |
| Operação | simples | exige instalação e hardware |

Não existe almoço grátis. Existe troca de restrições.

In [16]:
# ============================================================
# CRIAÇÃO DO VECTOR STORE COM OLLAMA EMBEDDINGS
# ============================================================

from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)

try:
    qtd_existente = vector_store._collection.count()
except Exception:
    qtd_existente = 0

if qtd_existente == 0:
    print("Índice vazio. Criando embeddings locais com Ollama...")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(PERSIST_DIR),
    )
else:
    print(f"Índice existente encontrado com {qtd_existente} chunks.")

print("Total de chunks indexados:", vector_store._collection.count())


Índice vazio. Criando embeddings locais com Ollama...
Total de chunks indexados: 179


### 3.1 Testando a busca semântica antes do LLM

Regra de engenharia:

> antes de culpar o LLM, verifique se o retriever trouxe contexto bom.

A célula abaixo ainda não gera resposta final. Ela só pergunta ao vector store quais trechos parecem relevantes.

In [17]:
# ============================================================
# TESTE DE BUSCA SEMÂNTICA
# ============================================================

pergunta_teste = "Qual é a política de investimento do fundo?"
docs = vector_store.similarity_search(pergunta_teste, k=4)

for i, doc in enumerate(docs, start=1):
    pagina = doc.metadata.get("page")
    pagina_humana = pagina + 1 if isinstance(pagina, int) else None
    print(f"\n--- Resultado {i} | Página {pagina_humana} ---")
    print(doc.page_content[:900])



--- Resultado 1 | Página 14 ---
regras gerais sobre fundos de investimento , o FUNDO não poderá deter mais de 20% (vinte por cento)  de seu 
patrimônio líquido em títulos ou valores mobiliários de emissão de empresas ligadas ao  ADMINISTRADOR  ao 
GESTOR, sem prejuízo das demais disposições regulamentares e da aprovação em assembleia geral quando 
caracterizada situação de conflito de interesses, nos termos da regulamentação específica. 
 
6.8.3. Caso o FUNDO invista preponderantemente em valores mobiliários, e em atendimento ao disposto nas 
regras gerais sobre fundos de investimento, o FUNDO poderá investir até 100% (cem por cento) do montante de 
seus recursos que possam ser investidos em cotas de Fundos Investidos administrados pelo ADMINISTRADOR, 
pelo GESTOR ou empresa a eles ligada na forma permitida na regulamentação específica.

--- Resultado 2 | Página 10 ---
5. CATEGORIA 
 
5.1. O FUNDO é constituído sob a forma de um fundo de investimento imobiliário, regido nos termos da 

## Parte 4 — RAG local com ChatOllama

Agora conectamos o contexto recuperado ao modelo de chat local.

O prompt precisa ser mais cuidadoso com modelo local pequeno. Em geral, vale ser explícito:

- responda apenas com o contexto;
- não invente regra;
- cite páginas quando possível;
- se não encontrar, diga que não encontrou;
- mantenha resposta objetiva.

Isso não elimina alucinação, mas reduz a chance e melhora a auditabilidade.

In [18]:
# ============================================================
# FUNÇÕES DO RAG LOCAL
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama


class FonteConsulta(BaseModel):
    documento: Optional[str] = Field(default=None)
    pagina: Optional[int] = Field(default=None)
    trecho: str


class RespostaRAGOllama(BaseModel):
    pergunta: str
    resposta: str
    fontes: list[FonteConsulta]
    paginas_consultadas: list[int]
    confianca: Literal["alta", "media", "baixa"]
    observacao: str


llm = ChatOllama(
    model=OLLAMA_CHAT_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """
Você é um especialista em análise de regulamentos financeiros.

Responda usando APENAS o contexto fornecido.
Se a resposta não estiver no contexto, diga claramente que não encontrou a informação no documento.
Sempre que possível, cite páginas e trechos de referência.
Não invente números, cláusulas, taxas ou regras.
Responda em português brasileiro, com clareza e objetividade.

Contexto:
{context}
"""),
    ("human", "{question}"),
])

chain_rag = prompt_rag | llm | StrOutputParser()


def pagina_humana(doc) -> Optional[int]:
    pagina = doc.metadata.get("page")
    if isinstance(pagina, int):
        return pagina + 1
    return None


def formatar_documentos(docs) -> str:
    partes = []
    for doc in docs:
        fonte = doc.metadata.get("source", "documento")
        pagina = pagina_humana(doc)
        partes.append(f"[Fonte: {fonte} | Página: {pagina}]\n{doc.page_content}")
    return "\n\n---\n\n".join(partes)


def extrair_fontes(docs) -> tuple[list[FonteConsulta], list[int]]:
    fontes: list[FonteConsulta] = []
    paginas: list[int] = []
    for doc in docs:
        pagina = pagina_humana(doc)
        if pagina is not None:
            paginas.append(pagina)
        fontes.append(FonteConsulta(
            documento=doc.metadata.get("source"),
            pagina=pagina,
            trecho=doc.page_content[:700],
        ))
    return fontes, sorted(set(paginas))


def inferir_confianca(docs) -> Literal["alta", "media", "baixa"]:
    if len(docs) >= 4:
        return "alta"
    if len(docs) >= 2:
        return "media"
    return "baixa"


def consultar_rag_local(pergunta: str, k: int = 5) -> dict:
    docs = vector_store.similarity_search(pergunta, k=k)
    contexto = formatar_documentos(docs)
    resposta_texto = chain_rag.invoke({"context": contexto, "question": pergunta})
    fontes, paginas = extrair_fontes(docs)
    resposta = RespostaRAGOllama(
        pergunta=pergunta,
        resposta=resposta_texto,
        fontes=fontes,
        paginas_consultadas=paginas,
        confianca=inferir_confianca(docs),
        observacao="Resposta gerada localmente com Ollama a partir dos trechos recuperados pelo RAG.",
    )
    return resposta.model_dump()


print("Funções do RAG local definidas.")


Funções do RAG local definidas.


In [19]:
# ============================================================
# TESTE DO RAG LOCAL
# ============================================================

resultado = consultar_rag_local(
    "Qual é a política de investimento do fundo? Cite as páginas usadas.",
    k=5,
)

print(resultado["resposta"])
print("\nPáginas consultadas:", resultado["paginas_consultadas"])
print("Confiança:", resultado["confianca"])


De acordo com o regulamento, a política de investimento do fundo é a seguinte:

"O FUNDO é uma comunhão de recursos captados por meio do sistema de distribuição de valores mobiliários, tendo por objeto o investimento em empreendimentos imobiliários na forma prevista na regulamentação aplicável, preponderantemente – assim entendido como mais de 50% (cinquenta por cento) do patrimônio líquido do FUNDO – através da aquisição de Certificados de Recebíveis Imobiliários (“CRI”), nos termos do item 6.2 e seus subitens abaixo."

Além disso, o fundo também pode investir em Letras de Crédito do Fundo (LCF), conforme mencionado na página 30 do regulamento.

É importante notar que existem limites para a aplicação por emissor e por modalidade de ativos financeiros, que devem ser observados.

Páginas consultadas: [10, 13, 14, 30]
Confiança: alta


## Parte 5 — Por que LangGraph aqui?

Poderíamos parar na função `consultar_rag_local`.

Mas a aula é sobre workflow. Então vamos tornar o fluxo explícito:

```text
WorkflowState
  pergunta
  intencao
  documentos
  resposta_rag
  resposta_final
```

E os nós:

| Nó | Responsabilidade |
|---|---|
| `classificar` | decidir se a pergunta é documental, taxas ou fora de escopo |
| `recuperar` | buscar trechos no ChromaDB |
| `gerar_resposta` | chamar o LLM local com o contexto |
| `fora_escopo` | responder sem consultar documento |
| `formatar` | montar saída final para o usuário |

Isso é mais verboso do que uma chain, mas é mais controlável. Em banco, controle importa.

In [20]:
# ============================================================
# ESTADO, CLASSIFICADOR E ROTEADOR DO LANGGRAPH
# ============================================================

from langgraph.graph import END, StateGraph


class WorkflowState(TypedDict):
    pergunta: str
    intencao: Optional[str]
    documentos: Optional[list]
    resposta_rag: Optional[dict]
    resposta_final: Optional[str]


prompt_classificador = ChatPromptTemplate.from_messages([
    ("system", """
Classifique a intenção da pergunta do usuário.

Responda com EXATAMENTE uma das opções:
- consulta_documental: pergunta sobre regras, política de investimento, administrador, gestor, cotistas, direitos ou obrigações do regulamento
- taxas: pergunta sobre taxas, remunerações, custos ou despesas
- fora_escopo: pergunta que não depende do regulamento financeiro

Não explique. Responda apenas com o rótulo.
"""),
    ("human", "Pergunta: {pergunta}"),
])

chain_classificador = prompt_classificador | llm | StrOutputParser()


def classificar_intencao(state: WorkflowState) -> WorkflowState:
    intencao = chain_classificador.invoke({"pergunta": state["pergunta"]}).strip().lower()
    if intencao not in {"consulta_documental", "taxas", "fora_escopo"}:
        intencao = "consulta_documental"
    print(f"[classificar] intenção: {intencao}")
    return {**state, "intencao": intencao}


def roteador(state: WorkflowState) -> str:
    return state.get("intencao") or "consulta_documental"


print("Estado, classificador e roteador definidos.")


Estado, classificador e roteador definidos.


In [21]:
# ============================================================
# NÓS DO WORKFLOW
# ============================================================

def node_recuperar_contexto(state: WorkflowState) -> WorkflowState:
    pergunta = state["pergunta"]
    intencao = state.get("intencao")
    k = 7 if intencao == "taxas" else 5

    print(f"[recuperar] buscando {k} trechos relevantes...")
    docs = vector_store.similarity_search(pergunta, k=k)
    return {**state, "documentos": docs}


def node_gerar_resposta(state: WorkflowState) -> WorkflowState:
    docs = state.get("documentos") or []
    pergunta = state["pergunta"]

    print("[gerar_resposta] chamando modelo local via Ollama...")
    contexto = formatar_documentos(docs)
    resposta_texto = chain_rag.invoke({"context": contexto, "question": pergunta})
    fontes, paginas = extrair_fontes(docs)

    resposta = RespostaRAGOllama(
        pergunta=pergunta,
        resposta=resposta_texto,
        fontes=fontes,
        paginas_consultadas=paginas,
        confianca=inferir_confianca(docs),
        observacao="Resposta gerada por workflow LangGraph com modelos locais via Ollama.",
    )

    return {**state, "resposta_rag": resposta.model_dump()}


def node_fora_escopo(state: WorkflowState) -> WorkflowState:
    print("[fora_escopo] pergunta não pertence ao domínio do regulamento.")
    return {
        **state,
        "resposta_final": (
            "Esta aula está limitada a perguntas sobre o regulamento financeiro indexado. "
            "Reformule a pergunta para consultar regras, política de investimento, taxas, "
            "obrigações ou informações documentais do fundo."
        ),
    }


def node_formatar_resposta(state: WorkflowState) -> WorkflowState:
    if state.get("resposta_final"):
        return state

    resposta = state.get("resposta_rag") or {}
    texto = resposta.get("resposta", "Não foi possível gerar resposta.")
    paginas = resposta.get("paginas_consultadas", [])
    confianca = resposta.get("confianca", "")
    observacao = resposta.get("observacao", "")

    partes = [texto]
    if paginas:
        partes.append(f"\nPáginas consultadas: {paginas}")
    if confianca:
        partes.append(f"Confiança heurística: {confianca}")
    if observacao:
        partes.append(f"Observação: {observacao}")

    return {**state, "resposta_final": "\n".join(partes)}


print("Nós do workflow definidos.")


Nós do workflow definidos.


In [22]:
# ============================================================
# COMPILAR O WORKFLOW LANGGRAPH
# ============================================================

builder = StateGraph(WorkflowState)

builder.add_node("classificar", classificar_intencao)
builder.add_node("recuperar", node_recuperar_contexto)
builder.add_node("gerar_resposta", node_gerar_resposta)
builder.add_node("fora_escopo", node_fora_escopo)
builder.add_node("formatar", node_formatar_resposta)

builder.set_entry_point("classificar")

builder.add_conditional_edges(
    "classificar",
    roteador,
    {
        "consulta_documental": "recuperar",
        "taxas": "recuperar",
        "fora_escopo": "fora_escopo",
    },
)

builder.add_edge("recuperar", "gerar_resposta")
builder.add_edge("gerar_resposta", "formatar")
builder.add_edge("fora_escopo", "formatar")
builder.add_edge("formatar", END)

workflow = builder.compile()

print("Workflow compilado com sucesso.")
print("Nós registrados:", list(workflow.get_graph().nodes.keys()))


Workflow compilado com sucesso.
Nós registrados: ['__start__', 'classificar', 'recuperar', 'gerar_resposta', 'fora_escopo', 'formatar', '__end__']


## Parte 6 — Testando cenários

Vamos testar três caminhos:

| Cenário | Pergunta | Caminho esperado |
|---|---|---|
| Consulta documental | política de investimento | recuperar -> gerar resposta |
| Taxas | taxas e remunerações | recuperar com mais contexto -> gerar resposta |
| Fora de escopo | cotação do dólar | resposta de limite de escopo |

Esse teste é importante porque LangGraph não é só uma forma bonita de chamar LLM. Ele explicita controle de fluxo.

In [23]:
# ============================================================
# HELPER DE EXECUÇÃO DO WORKFLOW
# ============================================================

def executar_workflow(pergunta: str) -> dict:
    print("=" * 100)
    print("PERGUNTA:", pergunta)
    print("=" * 100)
    estado_inicial: WorkflowState = {
        "pergunta": pergunta,
        "intencao": None,
        "documentos": None,
        "resposta_rag": None,
        "resposta_final": None,
    }
    resultado = workflow.invoke(estado_inicial)
    print("\nRESPOSTA FINAL:\n")
    print(resultado["resposta_final"])
    return resultado


In [24]:
# CENÁRIO 1 — consulta documental
resultado_1 = executar_workflow(
    "Qual é a política de investimento do fundo? Cite as páginas usadas."
)


PERGUNTA: Qual é a política de investimento do fundo? Cite as páginas usadas.
[classificar] intenção: consulta_documental
[recuperar] buscando 5 trechos relevantes...
[gerar_resposta] chamando modelo local via Ollama...

RESPOSTA FINAL:

De acordo com o regulamento, a política de investimento do fundo é a seguinte:

"O FUNDO é uma comunhão de recursos captados por meio do sistema de distribuição de valores mobiliários, tendo por objeto o investimento em empreendimentos imobiliários na forma prevista na regulamentação aplicável, preponderantemente – assim entendido como mais de 50% (cinquenta por cento) do patrimônio líquido do FUNDO – através da aquisição de Certificados de Recebíveis Imobiliários (“CRI”), nos termos do item 6.2 e seus subitens abaixo."

Além disso, o fundo também pode investir em Letras de Crédito do Fundo (LCF), conforme mencionado na página 30 do regulamento.

É importante notar que existem limites para a aplicação por emissor e por modalidade de ativos financeiros,

In [25]:
# CENÁRIO 2 — taxas e custos
resultado_2 = executar_workflow(
    "Liste as principais taxas, remunerações e custos previstos no regulamento."
)


PERGUNTA: Liste as principais taxas, remunerações e custos previstos no regulamento.
[classificar] intenção: taxas
[recuperar] buscando 7 trechos relevantes...
[gerar_resposta] chamando modelo local via Ollama...

RESPOSTA FINAL:

Com base nos trechos fornecidos do documento KNCR_Regulamento_05-2025.pdf, aqui estão as principais taxas, remunerações e custos previstos:

**Taxas:**

* Taxa Global (art. 10.4): contemplará quaisquer taxas de administração e gestão/performance e/ou taxa de ingresso/saída cobradas na realização de investimentos pelo FUNDO.
* Taxa de administração (art. 10.5): parcelas podem ser pagas diretamente pelo FUNDO aos prestadores de serviços contratados, desde que o somatório das parcelas não exceda o montante total da Taxa Global.

**Remunerações:**

* Remuneração paga aos prestadores de serviço (art. 1.6.3): deve ser realizada em condições de mercado, observadas as especificidades do serviço a ser prestado.
* Remuneração paga ao CUSTODIANTE (art. 2.2): não é espec

In [26]:
# CENÁRIO 3 — fora de escopo
resultado_3 = executar_workflow(
    "Qual é a cotação do dólar hoje?"
)


PERGUNTA: Qual é a cotação do dólar hoje?
[classificar] intenção: taxas
[recuperar] buscando 7 trechos relevantes...
[gerar_resposta] chamando modelo local via Ollama...

RESPOSTA FINAL:

Desculpe, mas não tenho acesso em tempo real a informações sobre a cotação do dólar ou de qualquer outra moeda. Além disso, como especialista em análise de regulamentos financeiros, meu conhecimento é baseado nas informações disponíveis até a data do meu último treinamento (dezembro de 2023).

Se você precisa de informação atualizada sobre a cotação do dólar, recomendo verificar fontes confiáveis de notícias financeiras ou sites de economia.

Páginas consultadas: [19, 23, 29, 31, 35, 37, 38]
Confiança heurística: alta
Observação: Resposta gerada por workflow LangGraph com modelos locais via Ollama.


## Parte 7 — Encapsulando em um módulo reutilizável

Até aqui, tudo funciona no notebook. Mas o padrão profissional é separar capacidade reutilizável de exploração didática.

A célula abaixo gera `rag_core_ollama.py`.

Ele não inclui o workflow LangGraph inteiro. Ele encapsula a capacidade RAG local:

- carregar índice;
- criar embeddings locais;
- recuperar trechos;
- gerar resposta com Ollama;
- devolver resposta estruturada.

Depois, esse módulo pode ser usado por MCP, ADK, API, CLI ou outro grafo LangGraph.

In [27]:
%%writefile rag_core_ollama.py
"""
rag_core_ollama.py

Capacidade especialista de RAG local usando Ollama.

Arquitetura:
    PDF -> chunks -> OllamaEmbeddings -> ChromaDB -> retriever -> ChatOllama -> resposta estruturada
"""

from __future__ import annotations

import shutil
from pathlib import Path
from typing import Literal, Optional

from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


class FonteConsulta(BaseModel):
    documento: Optional[str] = Field(default=None)
    pagina: Optional[int] = Field(default=None)
    trecho: str


class RespostaRAGOllama(BaseModel):
    pergunta: str
    resposta: str
    fontes: list[FonteConsulta]
    paginas_consultadas: list[int]
    confianca: Literal["alta", "media", "baixa"]
    observacao: str


class RAGLocalOllama:
    def __init__(
        self,
        pdf_path: str,
        persist_directory: str = "vectorstore/chroma_kncr_ollama",
        collection_name: str = "regulamento_kncr_ollama",
        ollama_base_url: str = "http://localhost:11434",
        embedding_model: str = "nomic-embed-text",
        chat_model: str = "llama3.2:3b",
        chunk_size: int = 900,
        chunk_overlap: int = 140,
        k: int = 5,
        recriar_indice: bool = False,
    ):
        self.pdf_path = Path(pdf_path)
        self.persist_directory = Path(persist_directory)
        self.collection_name = collection_name
        self.ollama_base_url = ollama_base_url
        self.embedding_model_name = embedding_model
        self.chat_model_name = chat_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.k = k
        self.recriar_indice = recriar_indice

        self.persist_directory.parent.mkdir(parents=True, exist_ok=True)

        self.embeddings = OllamaEmbeddings(
            model=self.embedding_model_name,
            base_url=self.ollama_base_url,
        )
        self.llm = ChatOllama(
            model=self.chat_model_name,
            base_url=self.ollama_base_url,
            temperature=0,
        )

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """
Você é um especialista em análise de regulamentos financeiros.

Responda usando APENAS o contexto fornecido.
Se a resposta não estiver no contexto, diga claramente que não encontrou a informação no documento.
Sempre que possível, cite fonte e página.
Não invente números, cláusulas, taxas ou regras.

Contexto:
{context}
"""),
            ("human", "{question}"),
        ])

        self.chain = self.prompt | self.llm | StrOutputParser()
        self.vector_store = self._preparar_vector_store()

    def _preparar_vector_store(self) -> Chroma:
        if self.recriar_indice and self.persist_directory.exists():
            shutil.rmtree(self.persist_directory)

        vector_store = Chroma(
            collection_name=self.collection_name,
            embedding_function=self.embeddings,
            persist_directory=str(self.persist_directory),
        )

        try:
            quantidade = vector_store._collection.count()
        except Exception:
            quantidade = 0

        if quantidade == 0:
            if not self.pdf_path.exists():
                raise FileNotFoundError(f"PDF não encontrado: {self.pdf_path}")
            paginas = PyPDFLoader(str(self.pdf_path)).load()
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap,
                length_function=len,
                separators=["\n\n", "\n", ". ", " ", ""],
            )
            chunks = splitter.split_documents(paginas)
            vector_store = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                collection_name=self.collection_name,
                persist_directory=str(self.persist_directory),
            )

        return vector_store

    @staticmethod
    def _pagina_humana(doc) -> Optional[int]:
        pagina = doc.metadata.get("page")
        if isinstance(pagina, int):
            return pagina + 1
        return None

    def _formatar_documentos(self, docs) -> str:
        partes = []
        for doc in docs:
            fonte = doc.metadata.get("source", "documento")
            pagina = self._pagina_humana(doc)
            partes.append(f"[Fonte: {fonte} | Página: {pagina}]\n{doc.page_content}")
        return "\n\n---\n\n".join(partes)

    def _fontes(self, docs) -> tuple[list[FonteConsulta], list[int]]:
        fontes: list[FonteConsulta] = []
        paginas: list[int] = []
        for doc in docs:
            pagina = self._pagina_humana(doc)
            if pagina is not None:
                paginas.append(pagina)
            fontes.append(FonteConsulta(
                documento=doc.metadata.get("source"),
                pagina=pagina,
                trecho=doc.page_content[:700],
            ))
        return fontes, sorted(set(paginas))

    @staticmethod
    def _inferir_confianca(docs) -> Literal["alta", "media", "baixa"]:
        if len(docs) >= 4:
            return "alta"
        if len(docs) >= 2:
            return "media"
        return "baixa"

    def buscar_trechos(self, pergunta: str, k: Optional[int] = None) -> dict:
        k_final = k or self.k
        docs = self.vector_store.similarity_search(pergunta, k=k_final)
        fontes, paginas = self._fontes(docs)
        return {
            "pergunta": pergunta,
            "k": k_final,
            "paginas_consultadas": paginas,
            "fontes": [fonte.model_dump() for fonte in fontes],
        }

    def consultar(self, pergunta: str, k: Optional[int] = None) -> dict:
        k_final = k or self.k
        docs = self.vector_store.similarity_search(pergunta, k=k_final)
        contexto = self._formatar_documentos(docs)
        resposta_texto = self.chain.invoke({"context": contexto, "question": pergunta})
        fontes, paginas = self._fontes(docs)
        resposta = RespostaRAGOllama(
            pergunta=pergunta,
            resposta=resposta_texto,
            fontes=fontes,
            paginas_consultadas=paginas,
            confianca=self._inferir_confianca(docs),
            observacao="Resposta gerada localmente com Ollama a partir dos trechos recuperados pelo RAG.",
        )
        return resposta.model_dump()


Writing rag_core_ollama.py


## Exercícios

### Exercício 1 — Troca de modelo

Troque `llama3.2:3b` por outro modelo disponível no Ollama.

Sugestões:

```bash
ollama pull mistral
ollama pull qwen2.5:7b
```

Compare:

- qualidade da resposta;
- velocidade;
- capacidade de seguir instruções;
- fidelidade às fontes.

### Exercício 2 — Nó de validação

Adicione um nó `validar_resposta` depois de `gerar_resposta`.

Ele deve verificar se a resposta final menciona páginas consultadas. Se não mencionar, acrescente uma observação de cautela.


## Encerramento

Nesta aula extra, você construiu um RAG local dentro de um workflow LangGraph.

Você viu:

1. Como usar Ollama como runtime local de modelos gratuitos.
2. Como gerar embeddings locais com `nomic-embed-text`.
3. Como persistir um índice ChromaDB sem depender de API externa.
4. Como usar `ChatOllama` para responder com base no contexto.
5. Como modelar um workflow LangGraph com classificação, recuperação, geração e formatação.
6. Como tratar perguntas fora de escopo.

